In [19]:
import numpy as np
import pandas as pd
import tensorflow as tf
import re
import string
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout


In [20]:
# Set random seed
tf.random.set_seed(42)
np.random.seed(42)

In [21]:
# Step 1: Load the downloaded IMDB dataset
file_path = r"IMDB Dataset.csv" 

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print(df.head())
print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Dataset loaded successfully!
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset shape: (50000, 2)

Columns: ['review', 'sentiment']


In [22]:
# Step 2: Prepare the dataset
# Expected columns: review and sentiment
df = df[['review', 'sentiment']].dropna()

df['review'] = df['review'].astype(str)
df['sentiment'] = df['sentiment'].str.lower().str.strip()

# Convert sentiment labels to 0 and 1
df['sentiment'] = df['sentiment'].map({'negative': 0, 'positive': 1})

df = df.dropna(subset=['sentiment'])
df['sentiment'] = df['sentiment'].astype(int)

print("\nClass distribution:")
print(df['sentiment'].value_counts())


Class distribution:
sentiment
1    25000
0    25000
Name: count, dtype: int64


In [23]:
# Step 3: Clean the reviews
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review'] = df['review'].apply(clean_text)

In [24]:
# Step 4: Split the dataset
X = df['review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 40000
Testing samples: 10000


In [25]:
# Step 5: Tokenize the text
vocab_size = 10000
max_length = 100

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token='<OOV>'
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad the sequences
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='pre',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='pre',
    truncating='post'
)

print("\nShape after padding:")
print("X_train:", X_train_pad.shape)
print("X_test:", X_test_pad.shape)


Shape after padding:
X_train: (40000, 100)
X_test: (10000, 100)


In [26]:
# Step 6: Build the LSTM model
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [27]:
# Step 7: Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()


Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 673,089 (2.57 MB)

 Trainable params: 673,089 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# Step 8: Train the model
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)


Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 24s 84ms/step - accuracy: 0.6293 - loss: 0.6015 - val_accuracy: 0.8270 - val_loss: 0.3859
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 82ms/step - accuracy: 0.8531 - loss: 0.3551 - val_accuracy: 0.8220 - val_loss: 0.4048
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 20s 81ms/step - accuracy: 0.8782 - loss: 0.3119 - val_accuracy: 0.8173 - val_loss: 0.4050
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 20s 81ms/step - accuracy: 0.8932 - loss: 0.2817 - val_accuracy: 0.8261 - val_loss: 0.4422
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 82ms/step - accuracy: 0.9107 - loss: 0.2421 - val_accuracy: 0.8224 - val_loss: 0.4331


In [29]:
# Step 9: Evaluate the model
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("\nTest Loss:", round(loss, 4))
print("Test Accuracy:", round(accuracy * 100, 2), "%")



Test Loss: 0.4459
Test Accuracy: 82.51 %


In [30]:
# Create a folder to store the model files
os.makedirs("saved_model", exist_ok=True)

# Save the trained LSTM model
model.save("saved_model/imdb_sentiment_lstm.keras")

# Save the tokenizer
with open("saved_model/tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!


In [31]:
# Step 10: Predict sentiment for a new review
def predict_sentiment(review):
    cleaned_review = clean_text(review)

    sequence = tokenizer.texts_to_sequences([cleaned_review])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding='pre',
        truncating='post'
    )

    prediction = float(model.predict(padded, verbose=0)[0][0])

    print("\nReview:", review)
    print("Prediction Probability:", round(prediction, 4))

    if prediction >= 0.5:
        print("Sentiment: Positive")
    else:
        print("Sentiment: Negative")


In [32]:
# Sample prediction
sample = "The movie was excellent!"
predict_sentiment(sample)


Review: The movie was excellent!
Prediction Probability: 0.7699
Sentiment: Positive
